# KAN-Vim: toàn bộ thí nghiệm trên Kaggle

**Trước khi chạy** (bên phải màn hình, mục *Session options* / *Settings*):
1. **Accelerator → GPU T4 x2**
2. **Internet → On** (cần xác minh số điện thoại cho tài khoản Kaggle)
3. **Add Input** → dataset `kan-vim` (file `kan-vim-v3.zip` bạn đã upload)

**Cách chạy:** bấm **Save Version → Save & Run All (Commit)**. Kaggle chạy nền, tắt máy cũng được.
Notebook tự dừng an toàn sau ~11 giờ (Kaggle giới hạn 12 giờ/phiên). Nếu chưa xong: xem `HUONG_DAN_CHAY.md`, mục *Kaggle – chạy tiếp*.

Toàn bộ kế hoạch (9 bước, ~59 lần train) cần nhiều phiên; mỗi phiên chạy tiếp từ chỗ dừng, các bước quan trọng chạy trước.


In [ ]:
# 1. Chuẩn bị: lấy code, khôi phục kết quả của phiên trước (nếu có)
import glob, os, shutil, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.py', recursive=True) if 'kanvim' in os.listdir(os.path.dirname(p))]
assert src, 'Không thấy code: Add Input -> dataset kan-vim'
CODE = '/tmp/kan-vim'
shutil.rmtree(CODE, ignore_errors=True); shutil.copytree(src[0], CODE)
print('code:', src[0], '->', CODE)

OUT = '/kaggle/working/results_plan'
prev = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/results_plan/plan.json', recursive=True)]
if prev and not os.path.exists(OUT):
    shutil.copytree(prev[0], OUT)
    print('khôi phục kết quả phiên trước từ', prev[0])
print('đã có:', sorted(os.listdir(OUT)) if os.path.exists(OUT) else 'chưa có gì (phiên đầu tiên)')


In [ ]:
# 2. Chạy toàn bộ 9 bước (tự bỏ qua phần đã xong, tự dừng an toàn sau 11 giờ)
!cd /tmp/kan-vim && python scripts/run_plan.py --profile kaggle --stop_after_hours 11     --out /kaggle/working/results_plan --data_root /tmp/data --ckpt /tmp/vim_t.pth


In [ ]:
# 3. Xem báo cáo (chạy được cả khi mới xong một phần)
import os
from IPython.display import Markdown, Image, display
R = '/kaggle/working/results_plan'
if os.path.exists(f'{R}/BAO_CAO.md'):
    display(Markdown(open(f'{R}/BAO_CAO.md', encoding='utf-8').read().split('## 5. Hình')[0]))
    for f in ['fig_lr_search.png', 'fig_curves.png', 'fig_test_top1.png', 'fig_acc_vs_gmacs.png']:
        if os.path.exists(f'{R}/{f}'):
            display(Image(f'{R}/{f}'))
else:
    print('Chưa có báo cáo')
